# QTDB 1-channel: Paper-A data + research-B model and multi-domain loss

This Kaggle notebook trains a controlled hybrid experiment:

- **Data protocol (A):** QT Database beats built from `pu1` P-wave onsets, record-level test split, 360 Hz, 512 samples, and baseline-wander noise from NSTDB.
- **Model (B):** the advanced 1D U-Net with HNF blocks, Bridge/FiLM timestep injection, skip connections, and bottleneck self-attention.
- **Loss (B):** mean Gaussian-noise L1 loss plus diffusion-SNR-weighted relative STFT magnitude loss on the reconstructed clean signal.

LUDB and the 12-lead pipeline are not used here. Two independent checkpoints are trained for the two BW protocols described in `paper.md`.


## 1. Setup


In [ ]:
from pathlib import Path
import gc
import json
import math
import random
import shutil
import subprocess
import sys
import time
import zipfile

try:
    import wfdb
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'wfdb'])
    import wfdb

import numpy as np
import pandas as pd
from scipy import signal
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
WORK_DIR = Path('/kaggle/working')
CACHE_DIR = WORK_DIR / 'qtdb_paper_a_cache_v2'
OUTPUT_DIR = WORK_DIR / 'qtdb_1ch_hybrid_checkpoints'
CACHE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Device:', DEVICE)


## 2. Reproducible configuration


In [ ]:
SEED = 1234
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Paper A: fixed test records.
TEST_RECORDS = [
    'sel123', 'sel233', 'sel302', 'sel307', 'sel820', 'sel853',
    'sel16420', 'sel16795', 'sele0106', 'sele0121', 'sel32',
    'sel49', 'sel14046', 'sel15814',
]

# Paper A preprocessing.
TARGET_FS = 360
BEAT_LENGTH = 512
MAX_UNPADDED_LENGTH = 496
INSERT_OFFSET = 16
P_ONSET_RETRACTION_MS = 40
REFLECT_PAD_MS = 100
CHANNELS = 1
TRAIN_FRACTION = 0.70
TRAIN_NOISE_TYPES = [1]
NOISE_LEVEL_VALUES = np.arange(20, 200, dtype=np.int16) / 100.0

# None = use all available Paper-A training records/beats.
MAX_TRAIN_RECORDS = None
MAX_CLEAN_BEATS = None

# Paper A training scale; model and objective are research B.
EPOCHS = 400
BATCH_SIZE = 96
LR = 1e-4
NUM_DIFFUSION_STEPS = 50
BETA_START = 1e-4
BETA_END = 0.5
BETA_SCHEDULE = 'quad'
GRAD_CLIP_NORM = 1.0
LR_STEP_SIZE = 150
LR_GAMMA = 0.1
VALID_EVERY = 1
METRIC_VALID_EVERY = 10
MAX_METRIC_VALID_BEATS = 256

# Research B multi-domain loss.
LAMBDA_TIME = 1.0
LAMBDA_FREQ = 0.1
STFT_N_FFT = 128
STFT_HOP_LENGTH = 64

# Research B architecture.
BASE_FEATS = 80
EMB_DIM = 128

# Stable setting for long Kaggle runs; avoids QueueFeederThread errors.
NUM_WORKERS = 0
PIN_MEMORY = False

CONFIG = {
    'experiment': 'paper_a_data__research_b_unet_multidomain',
    'data_protocol': 'paper_a_qtdb_pu1_nstdb_bw',
    'model_architecture': 'research_b_unet1d_hnf_bridge_attention',
    'objective': 'research_b_l1_mean_plus_snr_weighted_relative_stft_x0',
    'seed': SEED,
    'target_fs': TARGET_FS,
    'beat_length': BEAT_LENGTH,
    'epochs': EPOCHS,
    'batch_size': BATCH_SIZE,
    'lr': LR,
    'diffusion_steps': NUM_DIFFUSION_STEPS,
    'beta_start': BETA_START,
    'beta_end': BETA_END,
    'beta_schedule': BETA_SCHEDULE,
    'lambda_time': LAMBDA_TIME,
    'lambda_freq': LAMBDA_FREQ,
    'base_feats': BASE_FEATS,
    'emb_dim': EMB_DIM,
}
print(json.dumps(CONFIG, indent=2))


## 3. Build clean QTDB beats exactly in Paper-A style

A segment starts 40 ms before a `pu1` P-wave onset and ends at the next shifted P onset. It is retained only when it contains at most one annotated beat, is resampled to 360 Hz, and is no longer than 496 samples. Baseline is removed using the mean of the two endpoints, then the segment is inserted into a zero vector of length 512 starting at index 16.


In [ ]:
BEAT_SYMBOLS = {
    'N', 'L', 'R', 'B', 'A', 'a', 'J', 'S', 'V', 'r',
    'F', 'e', 'j', 'n', 'E', '/', 'f', 'Q', '?'
}


def get_qtdb_records():
    try:
        records = wfdb.get_record_list('qtdb')
    except Exception:
        records = wfdb.get_record_list('qtdb/1.0.0')
    return sorted(r for r in records if not r.endswith('/'))


def read_annotation(record_name, extension):
    return wfdb.rdann(record_name, extension, pn_dir='qtdb/1.0.0')


def p_wave_onsets(pu_annotation):
    symbols = np.asarray(pu_annotation.symbol)
    samples = np.asarray(pu_annotation.sample, dtype=np.int64)
    onsets = []
    for idx, symbol in enumerate(symbols):
        if symbol != '(':
            continue
        # In QTDB delineation annotations, '(' immediately precedes
        # the p marker for a P-wave onset.
        lookahead = symbols[idx + 1:min(idx + 4, len(symbols))]
        if 'p' in lookahead:
            onsets.append(int(samples[idx]))
    return np.asarray(onsets, dtype=np.int64)


def beat_locations(atr_annotation):
    return np.asarray([
        sample for sample, symbol in zip(atr_annotation.sample, atr_annotation.symbol)
        if symbol in BEAT_SYMBOLS
    ], dtype=np.int64)


def resample_with_reflect_padding(segment, source_fs, target_fs):
    if len(segment) < 3:
        return None
    pad_source = max(1, int(round(REFLECT_PAD_MS * source_fs / 1000.0)))
    pad_source = min(pad_source, len(segment) - 1)
    padded = np.pad(segment, (pad_source, pad_source), mode='reflect')
    target_length = int(round(len(padded) * target_fs / float(source_fs)))
    resampled = signal.resample(padded, target_length).astype(np.float32)
    pad_target = int(round(pad_source * target_fs / float(source_fs)))
    expected_length = int(round(len(segment) * target_fs / float(source_fs)))
    return resampled[pad_target:pad_target + expected_length]


def paper_a_segments_for_record(record_name):
    record = wfdb.rdrecord(record_name, pn_dir='qtdb/1.0.0', channels=[0])
    pu = read_annotation(record_name, 'pu1')
    atr = read_annotation(record_name, 'atr')
    ecg = record.p_signal[:, 0].astype(np.float32)
    fs = float(record.fs)

    shifted_onsets = p_wave_onsets(pu) - int(round(P_ONSET_RETRACTION_MS * fs / 1000.0))
    shifted_onsets = shifted_onsets[(shifted_onsets >= 0) & (shifted_onsets < len(ecg))]
    r_locations = beat_locations(atr)

    beats = []
    metadata = []
    for segment_index, (start, end) in enumerate(zip(shifted_onsets[:-1], shifted_onsets[1:])):
        if end <= start + 2:
            continue
        r_count = int(np.sum((r_locations >= start) & (r_locations < end)))
        if r_count > 1:
            continue
        segment = resample_with_reflect_padding(ecg[start:end], fs, TARGET_FS)
        if segment is None or len(segment) > MAX_UNPADDED_LENGTH:
            continue
        baseline = 0.5 * (float(segment[0]) + float(segment[-1]))
        segment = segment - baseline
        clean = np.zeros(BEAT_LENGTH, dtype=np.float32)
        clean[INSERT_OFFSET:INSERT_OFFSET + len(segment)] = segment
        if np.all(np.isfinite(clean)):
            beats.append(clean[:, None])
            metadata.append({
                'record': record_name,
                'segment_index': segment_index,
                'start_sample': int(start),
                'end_sample': int(end),
                'r_count': r_count,
                'unpadded_length': int(len(segment)),
            })
    return beats, metadata


def load_clean_beats(records, cache_stem, max_records=None, max_beats=None):
    cache_path = CACHE_DIR / f'{cache_stem}.npz'
    if cache_path.exists():
        cached = np.load(cache_path, allow_pickle=True)
        beats = cached['beats'].astype(np.float32)
        metadata = pd.DataFrame(cached['metadata'].tolist())
        print('Loaded cache:', cache_path, beats.shape)
        return beats, metadata

    selected_records = list(records)
    if max_records is not None:
        selected_records = selected_records[:max_records]

    all_beats, all_metadata = [], []
    for record_name in tqdm(selected_records, desc=cache_stem):
        try:
            beats, metadata = paper_a_segments_for_record(record_name)
            remaining = None if max_beats is None else max_beats - len(all_beats)
            if remaining is not None:
                beats, metadata = beats[:remaining], metadata[:remaining]
            all_beats.extend(beats)
            all_metadata.extend(metadata)
            print(f'{record_name}: {len(beats)} beats')
            if max_beats is not None and len(all_beats) >= max_beats:
                break
        except Exception as exc:
            print(f'Skip {record_name}: {type(exc).__name__}: {exc}')
        gc.collect()

    if not all_beats:
        raise RuntimeError('No QTDB beats were produced. Check Kaggle Internet and pu1/atr availability.')
    beat_array = np.stack(all_beats).astype(np.float32)
    np.savez_compressed(
        cache_path,
        beats=beat_array,
        metadata=np.asarray(all_metadata, dtype=object),
    )
    return beat_array, pd.DataFrame(all_metadata)


all_records = get_qtdb_records()
test_record_set = set(TEST_RECORDS)
training_records = [record for record in all_records if record not in test_record_set]
if MAX_TRAIN_RECORDS is not None:
    training_records = training_records[:MAX_TRAIN_RECORDS]

cache_limit_tag = f'r{MAX_TRAIN_RECORDS or "all"}_b{MAX_CLEAN_BEATS or "all"}'
clean_beats, clean_metadata = load_clean_beats(
    training_records,
    f'paper_a_train_records_{cache_limit_tag}',
    max_records=MAX_TRAIN_RECORDS,
    max_beats=MAX_CLEAN_BEATS,
)
print('Clean training-record beats:', clean_beats.shape)
display(clean_metadata.groupby('record').size().rename('beats').describe())


## 4. Split BW noise exactly as Paper A


In [ ]:
def load_bw_noise():
    record = wfdb.rdrecord('bw', pn_dir='nstdb/1.0.0')
    noise = record.p_signal.astype(np.float32)
    if int(record.fs) != TARGET_FS:
        target_length = int(round(len(noise) * TARGET_FS / float(record.fs)))
        noise = signal.resample(noise, target_length, axis=0).astype(np.float32)
    if noise.shape[1] < 2:
        raise RuntimeError('NSTDB bw must contain two channels for Paper-A protocols.')
    return noise


def training_noise_for_type(bw_noise, noise_type):
    midpoint = len(bw_noise) // 2
    if noise_type == 1:
        return bw_noise[:midpoint, 0].copy()
    if noise_type == 2:
        return bw_noise[:midpoint, 1].copy()
    raise ValueError('noise_type must be 1 or 2')


bw_noise = load_bw_noise()
training_noise = {noise_type: training_noise_for_type(bw_noise, noise_type)
                  for noise_type in TRAIN_NOISE_TYPES}
for noise_type, noise in training_noise.items():
    print(f'noise_type={noise_type}: train BW samples={len(noise)}')
del bw_noise
gc.collect()


## 5. Fixed Paper-A noisy/clean pairs

The remaining QTDB records are split 70/30 into train and validation with a fixed seed. Each clean beat receives one fixed BW location and one discrete level `r` from `0.20, 0.21, ..., 1.99`. Validation corruption never changes between epochs.


In [ ]:
all_indices = np.arange(len(clean_beats))
train_indices, val_indices = train_test_split(
    all_indices,
    train_size=TRAIN_FRACTION,
    random_state=SEED,
    shuffle=True,
)
print('Train beats:', len(train_indices), '| Validation beats:', len(val_indices))


def amplitude_range(values, eps=1e-8):
    return float(np.max(values) - np.min(values) + eps)


class FixedPaperANoisyDataset(Dataset):
    def __init__(self, clean_array, selected_indices, noise_signal, seed):
        self.clean_array = clean_array
        self.indices = np.asarray(selected_indices, dtype=np.int64)
        self.noise_signal = np.asarray(noise_signal, dtype=np.float32)
        rng = np.random.default_rng(seed)
        self.noise_starts = rng.integers(
            0, len(self.noise_signal) - BEAT_LENGTH, size=len(self.indices), dtype=np.int64
        )
        self.levels = rng.choice(NOISE_LEVEL_VALUES, size=len(self.indices)).astype(np.float32)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, item):
        clean = self.clean_array[self.indices[item]].astype(np.float32, copy=False)
        start = int(self.noise_starts[item])
        noise_patch = self.noise_signal[start:start + BEAT_LENGTH, None]
        alpha = float(self.levels[item]) * amplitude_range(clean) / amplitude_range(noise_patch)
        noisy = (clean + alpha * noise_patch).astype(np.float32)
        clean_tensor = torch.from_numpy(clean).permute(1, 0)
        noisy_tensor = torch.from_numpy(noisy).permute(1, 0)
        return clean_tensor, noisy_tensor


def build_datasets(noise_type):
    noise = training_noise[noise_type]
    train_dataset = FixedPaperANoisyDataset(
        clean_beats, train_indices, noise, seed=SEED + noise_type * 1000
    )
    val_dataset = FixedPaperANoisyDataset(
        clean_beats, val_indices, noise, seed=SEED + noise_type * 2000
    )
    return train_dataset, val_dataset


sample_dataset, _ = build_datasets(TRAIN_NOISE_TYPES[0])
sample_clean, sample_noisy = sample_dataset[0]
print('Tensor shapes:', tuple(sample_clean.shape), tuple(sample_noisy.shape))
print('Clean range:', float(sample_clean.max() - sample_clean.min()))
print('Noisy finite:', bool(torch.isfinite(sample_noisy).all()))
del sample_dataset, sample_clean, sample_noisy


## 6. Research-B advanced 1D U-Net


In [ ]:
class HNFBlockUNet(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_sizes=(3, 5, 9, 15)):
        super().__init__()
        self.multi_convs = nn.ModuleList([
            nn.Conv1d(in_channels, out_channels // len(kernel_sizes), k, padding=k // 2, padding_mode='reflect')
            for k in kernel_sizes
        ])
        self.agg_conv = nn.Conv1d(out_channels, out_channels, 1)
        self.half_inst_norm = nn.InstanceNorm1d(out_channels // 2)
        self.act = nn.ReLU(inplace=True)
        self.residual = nn.Conv1d(in_channels, out_channels, 1) if in_channels != out_channels else nn.Identity()

    def forward(self, x):
        out = torch.cat([conv(x) for conv in self.multi_convs], dim=1)
        out = self.agg_conv(out)
        half = out.shape[1] // 2
        out = torch.cat([self.half_inst_norm(out[:, :half, :]), out[:, half:, :]], dim=1)
        out = self.act(out)
        return out + self.residual(x)


class BridgeBlockUNet(nn.Module):
    def __init__(self, features, emb_dim=128):
        super().__init__()
        self.emb_dim = emb_dim
        self.film = nn.Sequential(nn.Linear(emb_dim, features * 2), nn.SiLU())

    def sinusoidal_embedding(self, x):
        x = x.view(-1)
        device = x.device
        half_dim = self.emb_dim // 2
        emb = math.log(10000) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=device) * -emb)
        emb = x.unsqueeze(-1) * emb.unsqueeze(0)
        return torch.cat([torch.sin(emb), torch.cos(emb)], dim=-1)

    def forward(self, x, alpha_bar):
        emb = self.sinusoidal_embedding(alpha_bar)
        scale, shift = self.film(emb).chunk(2, dim=1)
        return x * (1 + scale.unsqueeze(-1)) + shift.unsqueeze(-1)


class SelfAttention1D(nn.Module):
    def __init__(self, channels, num_heads=4):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = channels // num_heads
        assert self.head_dim * num_heads == channels
        self.qkv = nn.Conv1d(channels, channels * 3, kernel_size=1)
        self.proj = nn.Conv1d(channels, channels, kernel_size=1)

    def forward(self, x):
        batch, channels, length = x.shape
        qkv = self.qkv(x).reshape(batch, 3, self.num_heads, self.head_dim, length)
        q, k, v = qkv[:, 0], qkv[:, 1], qkv[:, 2]
        attn = torch.matmul(q.transpose(-2, -1), k) / (self.head_dim ** 0.5)
        attn = torch.softmax(attn, dim=-1)
        out = torch.matmul(attn, v.transpose(-2, -1)).transpose(-2, -1)
        return self.proj(out.reshape(batch, channels, length))


class UNet1D(nn.Module):
    def __init__(self, in_channels=2, base_channels=80, emb_dim=128, out_channels=1):
        super().__init__()
        self.enc1 = HNFBlockUNet(in_channels, base_channels)
        self.bridge1 = BridgeBlockUNet(base_channels, emb_dim)
        self.down1 = nn.Conv1d(base_channels, base_channels * 2, kernel_size=4, stride=2, padding=1)
        self.enc2 = HNFBlockUNet(base_channels * 2, base_channels * 2)
        self.bridge2 = BridgeBlockUNet(base_channels * 2, emb_dim)
        self.down2 = nn.Conv1d(base_channels * 2, base_channels * 4, kernel_size=4, stride=2, padding=1)
        self.enc3 = HNFBlockUNet(base_channels * 4, base_channels * 4)
        self.bridge3 = BridgeBlockUNet(base_channels * 4, emb_dim)
        self.down3 = nn.Conv1d(base_channels * 4, base_channels * 8, kernel_size=4, stride=2, padding=1)
        self.enc4 = HNFBlockUNet(base_channels * 8, base_channels * 8)
        self.bridge4 = BridgeBlockUNet(base_channels * 8, emb_dim)
        self.attn = SelfAttention1D(base_channels * 8)
        self.up4 = nn.ConvTranspose1d(base_channels * 8, base_channels * 4, kernel_size=4, stride=2, padding=1)
        self.dec4 = HNFBlockUNet(base_channels * 8, base_channels * 4)
        self.up3 = nn.ConvTranspose1d(base_channels * 4, base_channels * 2, kernel_size=4, stride=2, padding=1)
        self.dec3 = HNFBlockUNet(base_channels * 4, base_channels * 2)
        self.up2 = nn.ConvTranspose1d(base_channels * 2, base_channels, kernel_size=4, stride=2, padding=1)
        self.dec2 = HNFBlockUNet(base_channels * 2, base_channels)
        self.final = nn.Conv1d(base_channels, out_channels, kernel_size=1)

    def forward(self, x, cond, noise_scale):
        inp = torch.cat([x, cond], dim=1)
        e1 = self.bridge1(self.enc1(inp), noise_scale)
        e2 = self.bridge2(self.enc2(self.down1(e1)), noise_scale)
        e3 = self.bridge3(self.enc3(self.down2(e2)), noise_scale)
        e4 = self.bridge4(self.enc4(self.down3(e3)), noise_scale)
        e4 = self.attn(e4)
        d4 = self.dec4(torch.cat([self.up4(e4), e3], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e2], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e1], dim=1))
        return self.final(d2)


## 7. Conditional DDPM with research-B multi-domain loss


In [ ]:
def stft_magnitude_loss(prediction, target, continuous_sqrt_alpha):
    batch, channels, length = prediction.shape
    prediction = prediction.reshape(batch * channels, length)
    target = target.reshape(batch * channels, length)
    window = torch.hann_window(STFT_N_FFT, device=prediction.device)
    prediction_stft = torch.stft(
        prediction, n_fft=STFT_N_FFT, hop_length=STFT_HOP_LENGTH,
        window=window, return_complex=True
    )
    target_stft = torch.stft(
        target, n_fft=STFT_N_FFT, hop_length=STFT_HOP_LENGTH,
        window=window, return_complex=True
    )

    prediction_mag = torch.abs(prediction_stft).reshape(batch, channels, *prediction_stft.shape[-2:])
    target_mag = torch.abs(target_stft).reshape(batch, channels, *target_stft.shape[-2:])
    reduce_dims = (1, 2, 3)
    spectral_error = torch.mean((prediction_mag - target_mag) ** 2, dim=reduce_dims)
    target_power = torch.mean(target_mag ** 2, dim=reduce_dims).clamp_min(1e-6)
    relative_error = spectral_error / target_power

    # x0 reconstruction is ill-conditioned at very noisy timesteps. Weight its
    # spectral supervision by diffusion SNR and cap the low-noise weight at 1.
    alpha_bar = continuous_sqrt_alpha.reshape(batch) ** 2
    diffusion_snr = alpha_bar / torch.clamp(1.0 - alpha_bar, min=1e-6)
    frequency_weight = torch.clamp(diffusion_snr, max=1.0)
    weighted_loss = torch.mean(relative_error * frequency_weight)
    return weighted_loss, torch.mean(relative_error), torch.mean(frequency_weight)


def make_beta_schedule(schedule_name, num_steps, start, end):
    if schedule_name == 'linear':
        return torch.linspace(start, end, num_steps)
    if schedule_name == 'quad':
        return torch.linspace(start ** 0.5, end ** 0.5, num_steps) ** 2
    if schedule_name == 'sigmoid':
        values = torch.linspace(-6, 6, num_steps)
        return torch.sigmoid(values) * (end - start) + start
    raise ValueError(schedule_name)


class DDPM(nn.Module):
    def __init__(self, base_model):
        super().__init__()
        self.model = base_model
        self.num_steps = NUM_DIFFUSION_STEPS
        betas = make_beta_schedule(
            BETA_SCHEDULE, NUM_DIFFUSION_STEPS, BETA_START, BETA_END
        )
        alphas = 1.0 - betas
        alphas_cumprod = torch.cumprod(alphas, dim=0)
        alphas_cumprod_prev = torch.cat([torch.ones(1), alphas_cumprod[:-1]])
        continuous_boundaries = torch.sqrt(torch.cat([torch.ones(1), alphas_cumprod]))
        posterior_variance = betas * (1.0 - alphas_cumprod_prev) / (1.0 - alphas_cumprod)

        self.register_buffer('betas', betas.float())
        self.register_buffer('alphas_cumprod', alphas_cumprod.float())
        self.register_buffer('alphas_cumprod_prev', alphas_cumprod_prev.float())
        self.register_buffer('continuous_boundaries', continuous_boundaries.float())
        self.register_buffer('posterior_variance', posterior_variance.float())
        self.register_buffer(
            'posterior_log_variance_clipped',
            torch.log(torch.clamp(posterior_variance, min=1e-20)).float(),
        )
        self.register_buffer(
            'posterior_mean_coef1',
            (betas * torch.sqrt(alphas_cumprod_prev) / (1.0 - alphas_cumprod)).float(),
        )
        self.register_buffer(
            'posterior_mean_coef2',
            ((1.0 - alphas_cumprod_prev) * torch.sqrt(alphas) /
             (1.0 - alphas_cumprod)).float(),
        )

    def training_losses(self, clean, noisy_condition):
        batch = clean.shape[0]
        timestep = torch.randint(0, self.num_steps, (batch,), device=clean.device)
        lower = self.continuous_boundaries[timestep]
        upper = self.continuous_boundaries[timestep + 1]
        continuous = lower + torch.rand(batch, device=clean.device) * (upper - lower)
        continuous_3d = continuous.view(batch, 1, 1)

        gaussian_noise = torch.randn_like(clean)
        x_t = (
            continuous_3d * clean
            + torch.sqrt(torch.clamp(1.0 - continuous_3d ** 2, min=0.0)) * gaussian_noise
        )
        predicted_noise = self.model(x_t, noisy_condition, continuous.view(batch, 1))

        # Research B time-domain objective, normalized per element so its
        # scale is independent of batch size and ECG window length.
        loss_time = F.l1_loss(predicted_noise, gaussian_noise, reduction='mean')

        # Use the exact continuous coefficient that generated x_t.
        x0_prediction = (
            x_t
            - torch.sqrt(torch.clamp(1.0 - continuous_3d ** 2, min=0.0)) * predicted_noise
        ) / torch.clamp(continuous_3d, min=1e-6)
        loss_frequency, frequency_unweighted, frequency_weight = stft_magnitude_loss(
            x0_prediction, clean, continuous
        )
        total = LAMBDA_TIME * loss_time + LAMBDA_FREQ * loss_frequency
        return {
            'total': total,
            'time_mean': loss_time,
            'frequency': loss_frequency,
            'frequency_unweighted': frequency_unweighted,
            'frequency_weight': frequency_weight,
            'weighted_frequency': LAMBDA_FREQ * loss_frequency,
        }

    def forward(self, clean, noisy_condition):
        return self.training_losses(clean, noisy_condition)['total']

    def q_posterior(self, x_start, x_t, timestep):
        coef1 = self.posterior_mean_coef1[timestep].view(-1, 1, 1)
        coef2 = self.posterior_mean_coef2[timestep].view(-1, 1, 1)
        mean = coef1 * x_start + coef2 * x_t
        log_variance = self.posterior_log_variance_clipped[timestep].view(-1, 1, 1)
        return mean, log_variance

    @torch.no_grad()
    def sample(self, condition, num_shots=1):
        output_sum = torch.zeros_like(condition)
        batch = condition.shape[0]
        for _ in range(num_shots):
            x = torch.randn_like(condition)
            for step in reversed(range(self.num_steps)):
                timestep = torch.full((batch,), step, device=x.device, dtype=torch.long)
                noise_level = self.continuous_boundaries[step + 1].expand(batch, 1)
                predicted_noise = self.model(x, condition, noise_level)
                alpha_bar = self.alphas_cumprod[step]
                x0_prediction = (
                    x - torch.sqrt(1.0 - alpha_bar) * predicted_noise
                ) / torch.sqrt(alpha_bar)
                posterior_mean, posterior_log_variance = self.q_posterior(
                    x0_prediction, x, timestep
                )
                if step > 0:
                    x = posterior_mean + torch.exp(0.5 * posterior_log_variance) * torch.randn_like(x)
                else:
                    x = posterior_mean
            output_sum += x
        return output_sum / num_shots


## 8. Validation metrics and checkpoint helpers


In [ ]:
def batch_metrics(clean, denoised):
    clean_flat = clean.reshape(clean.shape[0], -1)
    denoised_flat = denoised.reshape(denoised.shape[0], -1)
    error_power = torch.sum((denoised_flat - clean_flat) ** 2, dim=1)
    signal_power = torch.sum(clean_flat ** 2, dim=1)
    cosine = F.cosine_similarity(clean_flat, denoised_flat, dim=1)
    snr = 10.0 * torch.log10((signal_power + 1e-8) / (error_power + 1e-8))
    return cosine, snr


@torch.no_grad()
def validate(model, validation_loader, compute_sampling_metrics=False):
    model.eval()
    total_per_element = []
    time_per_element = []
    frequency_losses = []
    cosines = []
    snrs = []
    sampled_beats = 0
    for clean, noisy in tqdm(validation_loader, desc='validation', leave=False):
        clean = clean.to(DEVICE)
        noisy = noisy.to(DEVICE)
        losses = model.training_losses(clean, noisy)
        total_per_element.append(losses['total'].item())
        time_per_element.append(losses['time_mean'].item())
        frequency_losses.append(losses['frequency'].item())
        if compute_sampling_metrics and sampled_beats < MAX_METRIC_VALID_BEATS:
            remaining = MAX_METRIC_VALID_BEATS - sampled_beats
            metric_clean = clean[:remaining]
            metric_noisy = noisy[:remaining]
            denoised = model.sample(metric_noisy, num_shots=1)
            cosine, snr = batch_metrics(metric_clean, denoised)
            cosines.extend(cosine.cpu().tolist())
            snrs.extend(snr.cpu().tolist())
            sampled_beats += len(metric_clean)
    return {
        'val_total_per_element': float(np.mean(total_per_element)),
        'val_time_mean': float(np.mean(time_per_element)),
        'val_frequency': float(np.mean(frequency_losses)),
        'val_cosine': float(np.mean(cosines)) if cosines else np.nan,
        'val_snr_out': float(np.mean(snrs)) if snrs else np.nan,
    }


def checkpoint_payload(model, optimizer, scheduler, noise_type, epoch, metrics):
    return {
        'state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'epoch': epoch,
        'noise_type': noise_type,
        'metrics': metrics,
        'config': CONFIG,
    }


## 9. Train one independent model per Paper-A noise protocol


In [ ]:
def train_one_noise_type(noise_type):
    print(f'\n=== noise_type={noise_type} ===')
    train_dataset, val_dataset = build_datasets(noise_type)
    generator = torch.Generator().manual_seed(SEED + noise_type)
    train_loader = DataLoader(
        train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True,
        num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, generator=generator,
    )
    val_loader = DataLoader(
        val_dataset, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
    )

    base_model = UNet1D(
        in_channels=2, base_channels=BASE_FEATS,
        emb_dim=EMB_DIM, out_channels=1,
    ).to(DEVICE)
    model = DDPM(base_model).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    scheduler = torch.optim.lr_scheduler.StepLR(
        optimizer, step_size=LR_STEP_SIZE, gamma=LR_GAMMA
    )

    best_validation = float('inf')
    history = []
    prefix = f'qtdb_1ch_Adata_Bmodel_multidomain_noise_type_{noise_type}'

    for epoch in range(1, EPOCHS + 1):
        epoch_start = time.time()
        model.train()
        total_per_element = []
        time_per_element = []
        frequency_losses = []

        for clean, noisy in tqdm(train_loader, desc=f'epoch {epoch}/{EPOCHS}', leave=False):
            clean = clean.to(DEVICE, non_blocking=True)
            noisy = noisy.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            losses = model.training_losses(clean, noisy)
            losses['total'].backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
            optimizer.step()

            total_per_element.append(losses['total'].item())
            time_per_element.append(losses['time_mean'].item())
            frequency_losses.append(losses['frequency'].item())

        scheduler.step()
        row = {
            'noise_type': noise_type,
            'epoch': epoch,
            'lr': optimizer.param_groups[0]['lr'],
            'train_total_per_element': float(np.mean(total_per_element)),
            'train_time_mean': float(np.mean(time_per_element)),
            'train_frequency': float(np.mean(frequency_losses)),
        }

        if epoch % VALID_EVERY == 0:
            validation_devices = [torch.cuda.current_device()] if DEVICE.type == 'cuda' else []
            with torch.random.fork_rng(devices=validation_devices):
                torch.manual_seed(SEED + noise_type * 10000)
                if DEVICE.type == 'cuda':
                    torch.cuda.manual_seed_all(SEED + noise_type * 10000)
                row.update(validate(
                    model,
                    val_loader,
                    compute_sampling_metrics=(epoch == 1 or epoch % METRIC_VALID_EVERY == 0),
                ))
        else:
            row.update({
                'val_total_per_element': np.nan,
                'val_time_mean': np.nan,
                'val_frequency': np.nan,
                'val_cosine': np.nan,
                'val_snr_out': np.nan,
            })
        row['seconds'] = round(time.time() - epoch_start, 2)
        history.append(row)
        print(row)

        payload = checkpoint_payload(model, optimizer, scheduler, noise_type, epoch, row)
        last_path = OUTPUT_DIR / f'{prefix}_last.pth'
        torch.save(payload, last_path)

        selection_metric = row['val_total_per_element']
        if np.isfinite(selection_metric) and selection_metric < best_validation:
            best_validation = selection_metric
            best_path = OUTPUT_DIR / f'{prefix}_best.pth'
            torch.save(payload, best_path)
            print('Saved best:', best_path.name)

        pd.DataFrame(history).to_csv(OUTPUT_DIR / f'{prefix}_training_log.csv', index=False)
        if DEVICE.type == 'cuda':
            torch.cuda.empty_cache()
        gc.collect()

    return pd.DataFrame(history)


## 10. Run training

Kaggle `Save Version -> Save & Run All` can continue on Kaggle after the browser is closed. Train one noise type at a time if the 400-epoch run approaches the session time limit by setting `TRAIN_NOISE_TYPES = [1]`, then `[2]` in a second version.


In [ ]:
all_history = []
for current_noise_type in TRAIN_NOISE_TYPES:
    current_history = train_one_noise_type(current_noise_type)
    all_history.append(current_history)

combined_history = pd.concat(all_history, ignore_index=True)
display(combined_history.tail())


## 11. Training curves and output checks


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for noise_type, group in combined_history.groupby('noise_type'):
    axes[0].plot(group['epoch'], group['train_time_mean'], label=f'train type {noise_type}')
    axes[0].plot(group['epoch'], group['val_time_mean'], '--', label=f'val type {noise_type}')
    axes[1].plot(group['epoch'], group['val_frequency'], label=f'type {noise_type}')
    axes[2].plot(group['epoch'], group['val_cosine'], label=f'type {noise_type}')
axes[0].set_title('Noise-prediction L1 mean')
axes[1].set_title('STFT magnitude loss')
axes[2].set_title('Validation cosine, 1-shot DDPM')
for axis in axes:
    axis.set_xlabel('Epoch')
    axis.grid(alpha=0.25)
    axis.legend()
plt.tight_layout()
curve_path = OUTPUT_DIR / 'training_curves.png'
plt.savefig(curve_path, dpi=160, bbox_inches='tight')
plt.show()

output_files = sorted(OUTPUT_DIR.glob('*'))
if not any(path.suffix == '.pth' for path in output_files):
    raise RuntimeError('Training finished without a .pth checkpoint.')
for path in output_files:
    print(path.name, f'{path.stat().st_size / 1024**2:.2f} MB')


## 12. Package Kaggle outputs


In [ ]:
for checkpoint_path in OUTPUT_DIR.glob('*.pth'):
    shutil.copy2(checkpoint_path, WORK_DIR / checkpoint_path.name)

archive_path = WORK_DIR / 'qtdb_1ch_Adata_Bmodel_multidomain_checkpoints.zip'
with zipfile.ZipFile(archive_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(OUTPUT_DIR.glob('*')):
        archive.write(path, arcname=path.name)
print('Archive:', archive_path)

from IPython.display import FileLink, display
for path in sorted(WORK_DIR.glob('qtdb_1ch_Adata_Bmodel_multidomain*.pth')):
    display(FileLink(str(path)))
display(FileLink(str(archive_path)))


## 13. Checkpoint contract for evaluation

Evaluation must instantiate the same research-B U-Net (`in_channels=2`, `out_channels=1`, `BASE_FEATS=80`, `EMB_DIM=128`) and the same 50-step quadratic diffusion schedule. Load `checkpoint['state_dict']`; use the matching checkpoint for each `noise_type`. Test beats and the second-half BW channels must be generated with the same Paper-A preprocessing above.
